# 02 - Model Training
Three models in a leakage-safe pipeline (scaler inside the pipeline). Model and threshold are chosen on cross-validation within the training set only.

In [1]:
import sys; sys.path.append('..')
import warnings; warnings.filterwarnings('ignore')
%matplotlib inline
import matplotlib.pyplot as plt, pandas as pd, numpy as np
from src.data_loader import load_data, split_data
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import average_precision_score, roc_auc_score
from src.preprocessing import get_models
from src.train import pick_threshold
X, y = load_data()
X_train, X_test, y_train, y_test = split_data(X, y)
print(X_train.shape, X_test.shape, y_train.mean().round(3), y_test.mean().round(3))

(455, 30) (114, 30) 0.374 0.368


## Cross-validated comparison
Repeating the 5-fold CV with several seeds shows how much the numbers move; with 455 training rows, differences between models are small.

In [2]:
rows = []
for seed in range(5):
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    for name, m in get_models(42).items():
        oof = cross_val_predict(m, X_train, y_train, cv=cv, method='predict_proba')[:,1]
        rows.append((name, seed, average_precision_score(y_train, oof)))
res = pd.DataFrame(rows, columns=['model','cv_seed','pr_auc'])
summary = res.groupby('model').pr_auc.agg(['mean','std']).round(4)
summary

,mean,std
model,,
Gradient Boosting,0.9844,0.0014
Logistic Regression,0.9921,0.0005
Random Forest,0.9865,0.0012


In [3]:
res.boxplot(by='model', column='pr_auc', figsize=(6,4)); plt.suptitle(''); plt.title('CV PR-AUC across 5 CV seeds'); plt.show()

## Threshold selection
For screening, a missed malignancy is costlier than a false alarm. The threshold is the highest value that reaches 98% recall on out-of-fold **training** scores. It is never tuned on the test set.

In [4]:
model = get_models(42)['Logistic Regression']
oof = cross_val_predict(model, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=42), method='predict_proba')[:,1]
thr = pick_threshold(y_train, oof)
print('Tuned threshold:', round(thr,3))

Tuned threshold: 0.284


## Final training
`python -m src.train` fits all models, saves the best by CV PR-AUC to `models/model.joblib`, and writes `reports/model_comparison.csv`.

In [5]:
pd.read_csv('../reports/model_comparison.csv')

,model,cv_pr_auc,cv_roc_auc,test_pr_auc,test_roc_auc,threshold_tuned,recall@0.5,precision@0.5,FN@0.5,FP@0.5,recall@tuned,precision@tuned,FN@tuned,FP@tuned
0,Logistic Regression,0.9941,0.9950,0.9931,0.9954,0.2845,0.9524,0.9756,2,1,0.9524,0.9524,2,2
1,Random Forest,0.9878,0.9891,0.9952,0.9970,0.2467,0.9048,1.0000,4,0,0.9762,0.9111,1,4
2,Gradient Boosting,0.9893,0.9912,0.9920,0.9947,0.0686,0.9048,1.0000,4,0,0.9286,0.9512,3,2
